# Retail Chatbot RAG Offline

Notebook ini digunakan untuk membangun knowledge base chatbot customer service retail berbasis RAG secara offline.

Tahapan utama:
1. Menyusun struktur folder proyek
2. Menyiapkan dataset dari CSV
3. Preprocessing data
4. Membuat dokumen knowledge base
5. Chunking dokumen
6. Membuat embedding
7. Menyimpan output ke file `embeddings.npy`, `documents.pkl`, dan `metadata.pkl`

## Cell 1 â€” Install Library

Jalankan cell ini jika library belum tersedia di Kaggle Notebook.

In [1]:
# Jika dijalankan di Kaggle dan belum ada sentence-transformers, aktifkan baris berikut.
# Catatan: Jika internet Kaggle dimatikan, pastikan model sudah tersedia di environment/dataset Kaggle.

!pip install -q sentence-transformers

## Cell 2 â€” Import Library

In [2]:
import os
import re
import pickle
import numpy as np
import pandas as pd

# SentenceTransformer di-import nanti pada Cell 15 agar tahap preprocessing tetap bisa dicek
# meskipun library embedding belum terpasang.

## Cell 3 â€” Membuat Struktur Folder Otomatis

In [3]:
folders = [
    "data/raw",
    "data/processed",
    "data/embeddings",
    "knowledge_base",
    "models",
    "retrieval",
    "chatbot",
    "evaluation",
    "output"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Struktur folder berhasil dibuat.")

## Cell 4 â€” Load Dataset Knowledge Base

Pastikan file berikut sudah tersedia:

- `knowledge_base/products.csv`
- `knowledge_base/faq.csv`
- `knowledge_base/policy.csv`
- `knowledge_base/stores.csv`

In [4]:
products_path = "knowledge_base/products.csv"
faq_path = "knowledge_base/faq.csv"
policy_path = "knowledge_base/policy.csv"
stores_path = "knowledge_base/stores.csv"

required_files = [
    products_path,
    faq_path,
    policy_path,
    stores_path
]

for file_path in required_files:
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File belum ditemukan: {file_path}")

products = pd.read_csv(products_path)
faq = pd.read_csv(faq_path)
policy = pd.read_csv(policy_path)
stores = pd.read_csv(stores_path)

print("Dataset berhasil dimuat.")
print("Products :", products.shape)
print("FAQ      :", faq.shape)
print("Policy   :", policy.shape)
print("Stores   :", stores.shape)

## Cell 5 â€” Melihat Kolom Dataset

In [5]:
print("Kolom products:", list(products.columns))
print("Kolom faq     :", list(faq.columns))
print("Kolom policy  :", list(policy.columns))
print("Kolom stores  :", list(stores.columns))

## Cell 6 â€” Fungsi Preprocessing

Proses:
- lowercase
- hapus karakter aneh
- hapus spasi berlebih
- normalisasi kata sederhana

In [6]:
def clean_text(text):
    text = str(text)
    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


stopwords = {
    "yang", "dan", "atau", "di", "ke", "dari", "untuk",
    "ini", "itu", "ada", "apakah", "dengan", "pada",
    "dalam", "sebuah", "adalah"
}


def normalize_text(text):
    words = text.split()
    words = [word for word in words if word not in stopwords]
    return " ".join(words)


def preprocess_text(text):
    text = clean_text(text)
    text = normalize_text(text)
    return text

## Cell 7 â€” Contoh Preprocessing

In [7]:
sample_text = "Produk ini tersedia?"

print("Input  :", sample_text)
print("Output :", preprocess_text(sample_text))

## Cell 8 â€” Membuat Document dari Products

In [8]:
documents = []
metadata = []

for idx, row in products.iterrows():
    doc = f'''
Produk:
Nama Produk : {row.get('product_name', '')}
Kategori : {row.get('category', '')}
Brand : {row.get('brand', '')}
Harga : {row.get('price', '')}
Stok : {row.get('stock', '')}
Deskripsi : {row.get('description', '')}
'''

    doc = preprocess_text(doc)

    documents.append(doc)

    metadata.append({
        "source": "products",
        "row_index": int(idx),
        "id": row.get("product_id", idx)
    })

print("Document products selesai:", len(documents))

## Cell 9 â€” Membuat Document dari FAQ

In [9]:
start_count = len(documents)

for idx, row in faq.iterrows():
    doc = f'''
FAQ:
Pertanyaan : {row.get('question', '')}
Jawaban : {row.get('answer', '')}
'''

    doc = preprocess_text(doc)

    documents.append(doc)

    metadata.append({
        "source": "faq",
        "row_index": int(idx),
        "id": row.get("faq_id", idx)
    })

print("Document FAQ selesai:", len(documents) - start_count)

## Cell 10 â€” Membuat Document dari Policy

In [10]:
start_count = len(documents)

for idx, row in policy.iterrows():
    doc = f'''
Kebijakan:
Jenis Kebijakan : {row.get('policy_type', '')}
Judul : {row.get('title', '')}
Isi : {row.get('description', '')}
'''

    doc = preprocess_text(doc)

    documents.append(doc)

    metadata.append({
        "source": "policy",
        "row_index": int(idx),
        "id": row.get("policy_id", idx)
    })

print("Document policy selesai:", len(documents) - start_count)

## Cell 11 â€” Membuat Document dari Stores

In [11]:
start_count = len(documents)

for idx, row in stores.iterrows():
    doc = f'''
Cabang:
Nama Cabang : {row.get('store_name', '')}
Kota : {row.get('city', '')}
Alamat : {row.get('address', '')}
Jam Operasional : {row.get('opening_hours', '')}
Telepon : {row.get('phone', '')}
'''

    doc = preprocess_text(doc)

    documents.append(doc)

    metadata.append({
        "source": "stores",
        "row_index": int(idx),
        "id": row.get("store_id", idx)
    })

print("Document stores selesai:", len(documents) - start_count)

## Cell 12 â€” Cek Hasil Document

In [12]:
print("Total documents:", len(documents))
print("\nContoh document pertama:")
print(documents[0])
print("\nMetadata:")
print(metadata[0])

## Cell 13 â€” Chunking Document

In [13]:
def chunk_text(text, chunk_size=80, overlap=10):
    words = text.split()

    if len(words) <= chunk_size:
        return [text]

    chunks = []
    start = 0

    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)

        start = end - overlap

        if start < 0:
            start = 0

        if start >= len(words):
            break

    return chunks


chunk_documents = []
chunk_metadata = []

for doc, meta in zip(documents, metadata):
    chunks = chunk_text(doc, chunk_size=80, overlap=10)

    for chunk_id, chunk in enumerate(chunks):
        chunk_documents.append(chunk)

        new_meta = meta.copy()
        new_meta["chunk_id"] = chunk_id

        chunk_metadata.append(new_meta)

print("Total original documents:", len(documents))
print("Total chunk documents   :", len(chunk_documents))

## Cell 14 â€” Cek Hasil Chunking

In [14]:
print("Contoh chunk pertama:")
print(chunk_documents[0])

print("\nMetadata chunk pertama:")
print(chunk_metadata[0])

## Cell 15 â€” Load Model Embedding Offline

Model utama:
- `sentence-transformers/all-MiniLM-L6-v2`

Alternatif:
- `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`
- `BAAI/bge-small-en-v1.5`

Untuk proyek retail berbahasa Indonesia, model multilingual bisa dipilih jika tersedia.

In [15]:
try:
    from sentence_transformers import SentenceTransformer
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Library sentence-transformers belum terpasang. Jalankan Cell 1 terlebih dahulu, "
        "atau install manual dengan: pip install sentence-transformers"
    ) from exc

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

model = SentenceTransformer(MODEL_NAME)

print("Model embedding berhasil dimuat:")
print(MODEL_NAME)

## Cell 16 â€” Membuat Embedding

In [16]:
embeddings = model.encode(
    chunk_documents,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Embedding selesai dibuat.")
print("Embedding shape:", embeddings.shape)

## Cell 17 â€” Simpan Output Knowledge Base

In [17]:
np.save("data/processed/embeddings.npy", embeddings)

with open("data/processed/documents.pkl", "wb") as f:
    pickle.dump(chunk_documents, f)

with open("data/processed/metadata.pkl", "wb") as f:
    pickle.dump(chunk_metadata, f)

print("File berhasil disimpan:")
print("data/processed/embeddings.npy")
print("data/processed/documents.pkl")
print("data/processed/metadata.pkl")

## Cell 18 â€” Verifikasi File Output

In [18]:
output_files = [
    "data/processed/embeddings.npy",
    "data/processed/documents.pkl",
    "data/processed/metadata.pkl"
]

for file_path in output_files:
    if os.path.exists(file_path):
        size_kb = os.path.getsize(file_path) / 1024
        print(f"OK: {file_path} - {size_kb:.2f} KB")
    else:
        print(f"TIDAK DITEMUKAN: {file_path}")

## Cell 19 â€” Load Ulang Output untuk Pengecekan

In [19]:
loaded_embeddings = np.load("data/processed/embeddings.npy")

with open("data/processed/documents.pkl", "rb") as f:
    loaded_documents = pickle.load(f)

with open("data/processed/metadata.pkl", "rb") as f:
    loaded_metadata = pickle.load(f)

print("Loaded embeddings shape:", loaded_embeddings.shape)
print("Loaded documents       :", len(loaded_documents))
print("Loaded metadata        :", len(loaded_metadata))

print("\nContoh dokumen:")
print(loaded_documents[0])

print("\nContoh metadata:")
print(loaded_metadata[0])

## Cell 20 â€” Ringkasan Tahap 3

Output akhir Tahap 3:

```text
data/processed/
â”œâ”€â”€ embeddings.npy
â”œâ”€â”€ documents.pkl
â””â”€â”€ metadata.pkl
```

File ini akan digunakan pada Tahap 4 untuk membangun mesin retrieval atau pencarian dokumen paling relevan.

## Tahap 4 - Retrieval Engine


In [20]:
# ==========================================
# TAHAP 4 - RETRIEVAL ENGINE
# Load Embeddings & Documents
# ==========================================

import os
import pickle
import numpy as np

# Path file hasil Tahap 3
EMBEDDINGS_PATH = "data/processed/embeddings.npy"
DOCUMENTS_PATH = "data/processed/documents.pkl"
METADATA_PATH = "data/processed/metadata.pkl"

# Load embeddings
embeddings = np.load(EMBEDDINGS_PATH)

# Load documents
with open(DOCUMENTS_PATH, "rb") as f:
    documents = pickle.load(f)

# Load metadata
with open(METADATA_PATH, "rb") as f:
    metadata = pickle.load(f)

print("OK: Embeddings berhasil dimuat")
print("Jumlah embeddings:", embeddings.shape)
print("Jumlah documents:", len(documents))
print("Jumlah metadata:", len(metadata))


In [21]:
# ==========================================
# Build FAISS Index
# ==========================================

!pip install faiss-cpu -q

import faiss

# Pastikan tipe data float32
embeddings = embeddings.astype("float32")

# Ambil dimensi embedding
dimension = embeddings.shape[1]

# Buat FAISS index
index = faiss.IndexFlatL2(dimension)

# Masukkan embeddings ke index
index.add(embeddings)

print("OK: FAISS Index berhasil dibuat")
print("Jumlah vector dalam index:", index.ntotal)
print("Dimensi vector:", dimension)


In [22]:
# ==========================================
# Save FAISS Index
# ==========================================

os.makedirs("retrieval", exist_ok=True)

FAISS_INDEX_PATH = "retrieval/faiss_index.index"

faiss.write_index(index, FAISS_INDEX_PATH)

print("OK: FAISS index berhasil disimpan di:", FAISS_INDEX_PATH)


In [23]:
# ==========================================
# Load Embedding Model untuk Query
# ==========================================

from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(MODEL_NAME)

print("OK: Model embedding query berhasil dimuat")


In [24]:
# ==========================================
# Retrieve Function
# ==========================================

def retrieve(query, top_k=5):
    """
    Fungsi untuk mencari dokumen paling relevan
    berdasarkan pertanyaan user.
    """

    # Ubah pertanyaan user menjadi embedding
    query_embedding = embedding_model.encode([query])
    query_embedding = np.array(query_embedding).astype("float32")

    # Search ke FAISS
    distances, indices = index.search(query_embedding, top_k)

    results = []

    for rank, idx in enumerate(indices[0]):
        result = {
            "rank": rank + 1,
            "distance": float(distances[0][rank]),
            "document": documents[idx],
            "metadata": metadata[idx]
        }
        results.append(result)

    return results


In [25]:
# ==========================================
# Testing Retrieval
# ==========================================

query = "Apakah Indomie Goreng masih tersedia?"

results = retrieve(query, top_k=2)

print("Pertanyaan:", query)
print("=" * 80)

for item in results:
    print(f"Rank      : {item['rank']}")
    print(f"Distance  : {item['distance']}")
    print(f"Metadata  : {item['metadata']}")
    print("Document  :")
    print(item["document"])
    print("-" * 80)


## Tahap 5 - Model AI (Generator)

Tahap ini menggunakan model TinyLlama sebagai generator jawaban. Context jawaban diambil dari hasil retrieval FAISS pada Tahap 4.


### Cell 1 - Install Library


In [ ]:
# Library generator untuk menjalankan model Hugging Face di Kaggle.
# hf_xet membantu proses download model Hugging Face lebih cepat dan stabil.
!pip install -q transformers accelerate sentencepiece hf_xet


### Cell 2 - Import Library


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


### Cell 3 - Cek GPU T4


In [ ]:
print("CUDA tersedia:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU tidak terdeteksi. Di Kaggle, aktifkan Accelerator: GPU T4.")


### Cell 3b - Smoke Test GPU T4


In [ ]:
# Smoke test ringan untuk memastikan notebook benar-benar berjalan di GPU.
print("PyTorch version:", torch.__version__)
print("CUDA tersedia:", torch.cuda.is_available())

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print("GPU aktif:", gpu_name)
    print("CUDA version:", torch.version.cuda)

    # Operasi kecil di GPU untuk memastikan tensor dapat diproses di CUDA.
    x = torch.randn(512, 512, device="cuda")
    y = torch.matmul(x, x)
    torch.cuda.synchronize()
    print("Smoke test CUDA: OK")

    if "T4" in gpu_name.upper():
        print("Status GPU: OK - menggunakan T4")
    else:
        print("Status GPU: GPU aktif, tetapi bukan T4. Kaggle memberi:", gpu_name)
else:
    print("Smoke test CUDA: GAGAL - GPU tidak aktif. Aktifkan Accelerator GPU di Kaggle.")


### Cell 4 - Load Model TinyLlama


In [ ]:
GENERATOR_MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Perpanjang timeout download agar lebih tahan terhadap koneksi lambat di Kaggle/Hugging Face.
import os
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "600"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "600"

tokenizer = AutoTokenizer.from_pretrained(GENERATOR_MODEL_NAME)

# Tentukan device agar kompatibel di Kaggle GPU T4 maupun CPU lokal.
device = "cuda" if torch.cuda.is_available() else "cpu"
model_dtype = torch.float16 if device == "cuda" else torch.float32

# Tidak memakai device_map="auto" agar tidak error jika accelerate belum aktif di kernel.
model = AutoModelForCausalLM.from_pretrained(
    GENERATOR_MODEL_NAME,
    dtype=model_dtype
)

model = model.to(device)
model.eval()
model.config.use_cache = True


print("Model TinyLlama berhasil dimuat:", GENERATOR_MODEL_NAME)
print("Device model:", device)


### Cell 5 - Prompt Template


In [ ]:
SYSTEM_PROMPT = """
Anda adalah customer service toko retail.

Jawablah hanya berdasarkan informasi yang diberikan pada Context.
Jika informasi tidak ditemukan pada Context, katakan:
"Maaf, data tersebut belum tersedia pada basis pengetahuan."

Jangan membuat informasi sendiri.
Jawaban harus singkat, jelas, dan sopan.
""".strip()


### Cell 6 - Build Prompt


In [46]:
def extract_context_documents(retrieval_results, max_docs=2, max_chars_per_doc=300):
    """
    Mengubah output retrieve() menjadi daftar teks dokumen.
    Context sengaja dibatasi agar generate lebih cepat.
    """
    context_documents = []

    for item in retrieval_results[:max_docs]:
        if isinstance(item, dict) and "document" in item:
            document = str(item["document"])
        else:
            document = str(item)

        context_documents.append(document[:max_chars_per_doc])

    return context_documents


def build_prompt(question, context_documents):
    context = "\n\n".join(context_documents)

    prompt = f"""<|system|>
{SYSTEM_PROMPT}</s>
<|user|>
Jawab dalam Bahasa Indonesia berdasarkan Context berikut.

Context:
{context}

Pertanyaan:
{question}

Jawaban singkat:</s>
<|assistant|>
"""
    return prompt


### Cell 7 - Generate Answer


In [47]:
def generate_answer(question, context_documents, max_new_tokens=40):
    """
    Membuat jawaban berdasarkan pertanyaan dan context hasil retrieval.
    Dibuat sangat ringan agar lebih cepat untuk demo/live.
    """
    prompt = build_prompt(question, context_documents)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=768
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            repetition_penalty=1.05,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    answer = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    stop_markers = ["Context:", "Pertanyaan:", "Jawaban:", "<|user|>", "<|system|>", "</s>"]
    for marker in stop_markers:
        if marker in answer:
            answer = answer.split(marker)[0].strip()

    return answer


### Cell 8 - Hubungkan Generator dengan Retriever Tahap 4


In [48]:
def chatbot(question, top_k=2, use_llm=True):
    """
    Pipeline chatbot: pertanyaan user -> retrieve context -> generate jawaban.
    top_k default dibuat 2 agar respons lebih cepat.
    Jika CPU terlalu lambat, set use_llm=False untuk jawaban berbasis context.
    """
    retrieval_results = retrieve(question, top_k=top_k)
    context_documents = extract_context_documents(retrieval_results, max_docs=top_k)

    if not context_documents:
        return "Maaf, data tersebut belum tersedia pada basis pengetahuan."

    if not use_llm:
        return "Berdasarkan basis pengetahuan: " + context_documents[0]

    answer = generate_answer(question, context_documents)

    if not answer:
        return "Maaf, data tersebut belum tersedia pada basis pengetahuan."

    return answer


### Cell 9 - Uji Retrieval Dulu


In [49]:
question = "Apakah Indomie Goreng masih tersedia?"

retrieval_results = retrieve(question, top_k=2)

for item in retrieval_results:
    print(f"Rank      : {item['rank']}")
    print(f"Distance  : {item['distance']}")
    print(f"Metadata  : {item['metadata']}")
    print("Document  :")
    print(item["document"])
    print("-" * 80)


### Cell 10 - Uji Chatbot


In [50]:
question = "Apakah Indomie Goreng masih tersedia?"

response = chatbot(question, top_k=2)

print("Pertanyaan:", question)
print("Jawaban:")
print(response)


### Cell 11 - Mode Tanya Jawab Aman untuk Run All


In [51]:
# Mode tanya jawab sekali jalan agar aman saat Run All.
# Ubah isi variable question sesuai pertanyaan yang ingin diuji.
question = "Apakah Indomie Goreng masih tersedia?"

response = chatbot(question, top_k=2)

print("User:", question)
print("Bot:", response)


### Cell 12 - Contoh Pengujian Beberapa Pertanyaan


In [52]:
# Jalankan cell ini jika ingin menguji beberapa pertanyaan tanpa mode interaktif.
test_questions = [
    "Apakah Indomie Goreng masih tersedia?",
    "Jam operasional toko sampai jam berapa?",
    "Bagaimana cara retur barang?"
]

for question in test_questions:
    print("User:", question)
    print("Bot:", chatbot(question, top_k=2))
    print("-" * 80)


## Tahap 6 - Evaluasi & Pengujian

Tahap ini mengevaluasi retrieval, kualitas jawaban chatbot, waktu respons, dan menyimpan hasil evaluasi ke folder `output/`.


### Cell 1 - Import Library Evaluasi


In [ ]:
# Library ringan untuk evaluasi. Tidak membutuhkan koneksi internet.
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd


### Cell 2 - Daftar Test Case Evaluasi


In [ ]:
# Test case dibuat berdasarkan knowledge base retail.
# expected_keyword dipakai untuk evaluasi PASS/FAIL sederhana.
test_cases = [
    {"question": "Berapa harga Indomie Goreng?", "expected_keyword": "3500", "category": "produk"},
    {"question": "Apakah Indomie Goreng masih tersedia?", "expected_keyword": "indomie", "category": "stok"},
    {"question": "Berapa stok Aqua 600ml?", "expected_keyword": "180", "category": "stok"},
    {"question": "Produk Teh Botol Sosro termasuk kategori apa?", "expected_keyword": "minuman", "category": "produk"},
    {"question": "Apa merek dari Indomie Goreng?", "expected_keyword": "indomie", "category": "produk"},
    {"question": "Di mana alamat RetailMart Bogor?", "expected_keyword": "pajajaran", "category": "cabang"},
    {"question": "Apakah ada cabang RetailMart Bandung?", "expected_keyword": "bandung", "category": "cabang"},
    {"question": "Jam operasional toko sampai jam berapa?", "expected_keyword": "22", "category": "jam operasional"},
    {"question": "Bagaimana cara mengecek stok produk?", "expected_keyword": "aplikasi", "category": "faq"},
    {"question": "Apakah bisa belanja online?", "expected_keyword": "website", "category": "faq"},
    {"question": "Bagaimana kebijakan retur barang?", "expected_keyword": "7", "category": "retur"},
    {"question": "Apakah produk memiliki garansi?", "expected_keyword": "produsen", "category": "garansi"},
    {"question": "Bagaimana metode pembayaran diproses?", "expected_keyword": "pembayaran", "category": "pembayaran"},
    {"question": "Apakah barang dikirim sebelum pembayaran?", "expected_keyword": "sebelum", "category": "pengiriman"},
    {"question": "Berapa nomor telepon RetailMart Jakarta Selatan?", "expected_keyword": "021", "category": "cabang"},
    {"question": "Apa alamat RetailMart Bandung?", "expected_keyword": "asia afrika", "category": "cabang"},
    {"question": "Apakah tersedia diskon ulang tahun pelanggan?", "expected_keyword": "belum tersedia", "category": "unknown"},
    {"question": "Apakah toko menerima pembayaran bitcoin?", "expected_keyword": "belum tersedia", "category": "unknown"},
]

print("Jumlah test case:", len(test_cases))


### Cell 3 - Fungsi Evaluasi Chatbot


In [ ]:
def _stage6_get_retriever():
    """Ambil fungsi retrieval yang sudah dibuat pada Tahap 4."""
    if "retrieve" in globals() and callable(globals()["retrieve"]):
        return globals()["retrieve"]
    # TODO: Jika nama fungsi retrieval berbeda, sesuaikan bagian ini.
    raise NameError("Fungsi retrieve(question, top_k) belum ditemukan. Jalankan Tahap 4 terlebih dahulu.")


def _stage6_get_chatbot():
    """Ambil fungsi chatbot yang sudah dibuat pada Tahap 5."""
    if "chatbot" in globals() and callable(globals()["chatbot"]):
        return globals()["chatbot"]
    # TODO: Jika nama fungsi chatbot berbeda, sesuaikan bagian ini.
    raise NameError("Fungsi chatbot(question) belum ditemukan. Jalankan Tahap 5 terlebih dahulu.")


def _stage6_extract_docs(retrieval_results):
    """Ubah hasil retrieval menjadi list dokumen teks."""
    docs = []
    for item in retrieval_results or []:
        if isinstance(item, dict):
            docs.append(str(item.get("document", "")))
        else:
            docs.append(str(item))
    return docs


def _stage6_keyword_found(text, keyword):
    """Cek keyword dengan pendekatan case-insensitive sederhana."""
    text = str(text).lower()
    keyword = str(keyword).lower()
    if not keyword:
        return False
    return keyword in text


def evaluate_chatbot(test_cases, top_k=3, use_llm=True):
    """
    Evaluasi chatbot otomatis.
    Fungsi ini menjalankan retrieval, chatbot, menghitung waktu respons,
    mengecek expected keyword, dan mengembalikan DataFrame hasil evaluasi.
    """
    retriever = _stage6_get_retriever()
    chat_fn = _stage6_get_chatbot()
    rows = []

    for no, case in enumerate(test_cases, start=1):
        question = case.get("question", "")
        expected_keyword = case.get("expected_keyword", "")
        category = case.get("category", "uncategorized")
        started_at = time.perf_counter()

        retrieval_results = []
        answer = ""
        error = ""

        try:
            retrieval_results = retriever(question, top_k=top_k)
            try:
                answer = chat_fn(question, top_k=top_k, use_llm=use_llm)
            except TypeError:
                answer = chat_fn(question)
        except Exception as exc:
            error = str(exc)
            answer = "ERROR: " + error

        response_time = time.perf_counter() - started_at
        retrieved_docs = _stage6_extract_docs(retrieval_results)
        retrieved_text = " ".join(retrieved_docs)

        answer_has_keyword = _stage6_keyword_found(answer, expected_keyword)
        retrieval_has_keyword = _stage6_keyword_found(retrieved_text, expected_keyword)
        status = "PASS" if (answer_has_keyword or retrieval_has_keyword) else "FAIL"

        rows.append({
            "No": no,
            "Category": category,
            "Question": question,
            "Expected Keyword": expected_keyword,
            "Answer": answer,
            "Retrieved Top K": len(retrieved_docs),
            "Retrieval Has Keyword": retrieval_has_keyword,
            "Answer Has Keyword": answer_has_keyword,
            "Status": status,
            "Response Time (s)": round(response_time, 3),
            "Error": error,
            "Retrieved Context": retrieved_text[:1200],
        })

    return pd.DataFrame(rows)


### Cell 4 - Metrik Retrieval


In [ ]:
def calculate_retrieval_metrics(evaluation_df):
    """
    Hitung metrik retrieval sederhana berdasarkan keberadaan expected keyword
    pada context hasil retrieval.
    """
    if evaluation_df.empty:
        return {
            "top_k_accuracy": 0.0,
            "precision_at_k": 0.0,
            "recall_at_k": 0.0,
            "average_response_time": 0.0,
        }

    retrieval_hits = evaluation_df["Retrieval Has Keyword"].astype(bool)
    top_k_accuracy = retrieval_hits.mean()

    # Versi sederhana: setiap pertanyaan punya satu keyword target.
    precision_at_k = retrieval_hits.sum() / max(evaluation_df["Retrieved Top K"].sum(), 1)
    recall_at_k = retrieval_hits.mean()
    average_response_time = evaluation_df["Response Time (s)"].mean()

    return {
        "top_k_accuracy": round(float(top_k_accuracy), 4),
        "precision_at_k": round(float(precision_at_k), 4),
        "recall_at_k": round(float(recall_at_k), 4),
        "average_response_time": round(float(average_response_time), 4),
    }


### Cell 5 - Evaluasi Kualitas Jawaban LLM


In [ ]:
def add_answer_quality_columns(evaluation_df):
    """
    Evaluasi kualitas jawaban dengan aturan sederhana berbasis keyword dan context.
    Kolom yang dibuat: Relevansi, Kelengkapan, Akurasi, Hallucination.
    """
    df = evaluation_df.copy()

    relevansi = []
    kelengkapan = []
    akurasi = []
    hallucination = []

    fallback_text = "belum tersedia"

    for _, row in df.iterrows():
        answer = str(row.get("Answer", "")).lower()
        context = str(row.get("Retrieved Context", "")).lower()
        expected = str(row.get("Expected Keyword", "")).lower()

        is_unknown = row.get("Category") == "unknown"
        answer_has_expected = expected in answer if expected else False
        context_has_expected = expected in context if expected else False
        uses_fallback = fallback_text in answer

        relevan = bool(answer_has_expected or context_has_expected or (is_unknown and uses_fallback))
        lengkap = bool(len(answer.strip()) >= 15 and not answer.startswith("error:"))
        akurat = bool(answer_has_expected or (context_has_expected and row.get("Status") == "PASS") or (is_unknown and uses_fallback))

        # Rule sederhana: jika bukan pertanyaan unknown, jawaban dianggap aman bila keyword
        # ada di jawaban atau minimal ada di context retrieval.
        if is_unknown:
            halu = not uses_fallback
        else:
            halu = not (answer_has_expected or context_has_expected)

        relevansi.append(relevan)
        kelengkapan.append(lengkap)
        akurasi.append(akurat)
        hallucination.append(halu)

    df["Relevansi"] = relevansi
    df["Kelengkapan"] = kelengkapan
    df["Akurasi"] = akurasi
    df["Hallucination"] = hallucination

    return df


### Cell 6 - Jalankan dan Tampilkan Hasil Evaluasi


In [ ]:
# Jika generate LLM terlalu lama, ubah use_llm menjadi False.
USE_LLM_FOR_EVALUATION = False
EVALUATION_TOP_K = 3

evaluation_df = evaluate_chatbot(
    test_cases,
    top_k=EVALUATION_TOP_K,
    use_llm=USE_LLM_FOR_EVALUATION
)
evaluation_df = add_answer_quality_columns(evaluation_df)

display_columns = [
    "No", "Category", "Question", "Answer", "Status",
    "Response Time (s)", "Relevansi", "Akurasi", "Hallucination"
]
evaluation_df[display_columns]


### Cell 7 - Ringkasan Metrik Evaluasi


In [ ]:
retrieval_metrics = calculate_retrieval_metrics(evaluation_df)

evaluation_summary = {
    "jumlah_test": int(len(evaluation_df)),
    "jumlah_pass": int((evaluation_df["Status"] == "PASS").sum()),
    "jumlah_fail": int((evaluation_df["Status"] == "FAIL").sum()),
    "top_k_accuracy": retrieval_metrics["top_k_accuracy"],
    "precision_at_k": retrieval_metrics["precision_at_k"],
    "recall_at_k": retrieval_metrics["recall_at_k"],
    "average_response_time": retrieval_metrics["average_response_time"],
    "persentase_hallucination": round(float(evaluation_df["Hallucination"].mean() * 100), 2),
    "persentase_jawaban_relevan": round(float(evaluation_df["Relevansi"].mean() * 100), 2),
}

summary_df = pd.DataFrame([evaluation_summary])
summary_df


### Cell 8 - Simpan Hasil Evaluasi


In [ ]:
output_dir = Path("output")
output_dir.mkdir(parents=True, exist_ok=True)

results_path = output_dir / "evaluation_results.csv"
summary_path = output_dir / "evaluation_summary.txt"

evaluation_df.to_csv(results_path, index=False, encoding="utf-8")

summary_lines = [
    "Retail Customer Service Chatbot - Evaluation Summary",
    "=" * 60,
]
for key, value in evaluation_summary.items():
    summary_lines.append(f"{key}: {value}")

summary_path.write_text("\n".join(summary_lines), encoding="utf-8")

print("Hasil evaluasi berhasil disimpan:")
print("-", results_path)
print("-", summary_path)
